In [1]:
# ─────────────────────────────────────────────────────────────
# [실습 목표] 세 가지 방법으로 감성 분류 성능 비교
#
#  1. Zero-shot prompting  : 예시 없이 모델에게 바로 질문
#  2. Few-shot prompting   : 몇 가지 레이블된 예시를 프롬프트에 포함
#  3. Full Fine-tuning     : 모델 전체 가중치를 학습 데이터로 업데이트
#
# [비교 기준]
#  - 성능 (Accuracy)
#  - 자원 소모 (학습 시간, 업데이트된 파라미터 비율)
#
# [사용 모델] google/flan-t5-small
#  - T5 계열의 seq2seq(인코더-디코더) 소형 모델 (~77M 파라미터)
#  - 텍스트 → 텍스트 형태로 분류 결과를 출력
# ─────────────────────────────────────────────────────────────

In [2]:
import time
import torch
import pandas as pd
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,      # 토크나이저 + seq2seq 모델 로드
                           Seq2SeqTrainer, Seq2SeqTrainingArguments,  # 파인튜닝 학습기 및 학습 설정
                           DataCollatorForSeq2Seq)                    # seq2seq 배치 패딩 처리용 콜레이터

# GPU 사용 가능하면 cuda, 아니면 cpu
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'using devices: {device}')

c:\Users\Playdata\miniconda3\envs\llm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


using devices: cpu


In [3]:
# ── 감성 분류용 데이터셋 정의 ─────────────────────────────────
# 각 샘플: 영화 리뷰 텍스트(text) + 정답 레이블(label: positive / negative)

train_data = [   # 학습 데이터 15개
    {"text": "This movie was absolutely fantastic! The acting was superb.", "label": "positive"},
    {"text": "I wasted two hours of my life. Terrible plot and cheap CGI.", "label": "negative"},
    {"text": "A true masterpiece of cinema. Highly recommend it to everyone.", "label": "positive"},
    {"text": "Extremely boring and predictable. I fell asleep halfway through.", "label": "negative"},
    {"text": "The cinematography was beautiful, and the story was touching.", "label": "positive"},
    {"text": "Horrible acting and bad direction. Do not watch this film.", "label": "negative"},
    {"text": "Brilliant performance by the lead actor, a must-watch.", "label": "positive"},
    {"text": "The plot made no sense, and the characters were annoying.", "label": "negative"},
    {"text": "I loved the soundtrack and the emotional depth of this movie.", "label": "positive"},
    {"text": "Total waste of money. I regret watching this garbage.", "label": "negative"},
    {"text": "Very engaging storyline with great character development.", "label": "positive"},
    {"text": "Slow, dull, and lacks any real substance.", "label": "negative"},
    {"text": "An amazing adventure that kept me on the edge of my seat.", "label": "positive"},
    {"text": "A complete disappointment. The trailer was much better than the film.", "label": "negative"},
    {"text": "Wonderfully written and beautifully executed. A delight to watch.", "label": "positive"}
]

test_data = [   # 평가 데이터 5개 (세 방법의 정확도 측정에 사용)
    {"text": "The movie was mediocre, but the ending was spectacular!", "label": "positive"},
    {"text": "Worst movie I have seen this year. Avoid at all costs.", "label": "negative"},
    {"text": "A refreshing and delightful comedy that had me laughing all night.", "label": "positive"},
    {"text": "The acting was flat and the script felt extremely forced.", "label": "negative"},
    {"text": "An absolute gem of a film with outstanding visuals.", "label": "positive"}
]

print(f"Train dataset size: {len(train_data)}")
print(f"Test dataset size: {len(test_data)}")

Train dataset size: 15
Test dataset size: 5


In [4]:
model_name = 'google/flan-t5-small'   # 허깅페이스 허브 모델 ID

# 토크나이저: 텍스트 → 토큰 ID 변환 (모델과 동일한 어휘 사전 사용)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 모델: flan-t5-small (인코더-디코더 seq2seq 구조, 약 77M 파라미터)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model.to(device)   # 모델을 GPU 또는 CPU로 이동

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 8628.48it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
              (wo): 

In [5]:
# ── 모델 파라미터 수 확인 함수 ────────────────────────────────
# 전체 파라미터 수 / 학습 가능한 파라미터 수 / 비율(%) 반환
# → Full Fine-tuning vs PEFT 비교 시 "얼마나 많은 가중치를 업데이트하는가" 확인용

# 참고: 개별 파라미터 속성 확인 방법
# _, param = next(iter(model.named_parameters()))
# param.numel()        # 해당 레이어의 파라미터 개수
# param.requires_grad  # True면 학습(gradient 업데이트) 대상

def get_trainable_params(model):
    all_param = 0
    trainable_params = 0
    for _, param in model.named_parameters():   # 모델의 모든 파라미터 레이어 순회
        all_param += param.numel()              # 전체 파라미터 수 누적
        if param.requires_grad:                 # gradient 계산 대상(학습 가능) 파라미터만
            trainable_params += param.numel()
    return all_param, trainable_params, trainable_params / all_param * 100   # (전체, 학습가능, 비율%)

all_p, train_p, pct = get_trainable_params(model)
all_p, train_p, pct

(76961152, 76961152, 100.0)

In [6]:
# ── 텍스트 생성(추론) 함수 ────────────────────────────────────
# prompt를 받아 모델이 생성한 텍스트(예측 레이블)를 반환
def generate_prediction(prompt, model, tokenizer, max_new_tokens=10):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)        # 프롬프트 → 토큰 텐서, 디바이스 이동
    with torch.no_grad():                                              # 추론 시 gradient 계산 비활성화 (메모리 절약)
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)  # 최대 10토큰 생성
    pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()  # 토큰 ID → 문자열, 소문자 정리
    return pred_text

# ── 평가 루프 ────────────────────────────────────────────────
# test_data 전체를 순회하며 예측하고 Accuracy와 결과 DataFrame을 반환
def evaluate_model(model, tokenizer, test_data, prompt_type='zero_shot'):
    correct = 0
    results = []

    # few-shot 프롬프트에 포함할 예시 3개 (정답 레이블 포함된 시범 샘플)
    few_shot_examples = (
        "Review: Brilliant performance by the lead actor, a must-watch. \nSentiment: positive\n\n"
        "Review: The plot made no sense, and the characters were annoying. \nSentiment: negative\n\n"
        "Review: I loved the soundtrack and the emotional depth of this movie. \nSentiment: positive\n\n"
    )

    for item in test_data:
        text = item['text']
        true_label = item['label']

        # 프롬프트 방식에 따라 입력 구성
        if prompt_type == 'zero_shot':           # 예시 없이 바로 질문
            prompt = f'Review:{text} \nSentiment (positive or negative)'
        elif prompt_type == 'few_shot':          # 예시 3개를 앞에 붙여 질문
            prompt = few_shot_examples + f'Review:{text} \nSentiment (positive or negative)'
        else:                                    # fine-tuning 후 평가 시 동일 형식 사용
            prompt = f'Review:{text} \nSentiment (positive or negative)'

        pred_label = generate_prediction(prompt, model, tokenizer)

        # 모델 출력에서 positive/negative 키워드만 추출 (불필요한 문자 제거)
        if 'positive' in pred_label:
            cleaned_pred = 'positive'
        elif 'negative' in pred_label:
            cleaned_pred = 'negative'
        else:
            cleaned_pred = pred_label            # 해당 없으면 원본 유지

        is_correct = (cleaned_pred == true_label)
        if is_correct:
            correct += 1

        results.append({
            'Text': text,
            'True Label': true_label,
            'Pred Label': pred_label,
            'Cleaned Pred': cleaned_pred,
            'Correct': is_correct
        })

    accuracy = correct / len(test_data)          # 정확도 = 맞춘 샘플 수 / 전체 샘플 수
    return accuracy, pd.DataFrame(results)

#### Zero Shot Prompting

In [7]:
zero_shot_acc, zero_shot_df = evaluate_model(model, tokenizer, test_data, 'zero_shot')
print(f'zero shot acc : {zero_shot_acc}')
zero_shot_df

zero shot acc : 0.2


,Text,True Label,Pred Label,Cleaned Pred,Correct
0,"The movie was mediocre, but the ending was spe...",positive,the movie is a great movie. the story,the movie is a great movie. the story,False
1,Worst movie I have seen this year. Avoid at al...,negative,i have seen this movie for years and have never,i have seen this movie for years and have never,False
2,A refreshing and delightful comedy that had me...,positive,i loved this movie. it was a great,i loved this movie. it was a great,False
3,The acting was flat and the script felt extrem...,negative,negative,negative,True
4,An absolute gem of a film with outstanding vis...,positive,this is a great film. it is,this is a great film. it is,False


#### Few Shot Prompting

In [8]:
few_shot_acc, few_shot_df = evaluate_model(model, tokenizer, test_data, 'few_shot')
print(f'few shot acc : {few_shot_acc}')
few_shot_df

few shot acc : 1.0


,Text,True Label,Pred Label,Cleaned Pred,Correct
0,"The movie was mediocre, but the ending was spe...",positive,positive,positive,True
1,Worst movie I have seen this year. Avoid at al...,negative,negative,negative,True
2,A refreshing and delightful comedy that had me...,positive,positive,positive,True
3,The acting was flat and the script felt extrem...,negative,negative,negative,True
4,An absolute gem of a film with outstanding vis...,positive,positive,positive,True


#### 전체 매개변수 파인튜닝(Full Fine Tuning)

In [9]:
# 1. 데이터 토크나이징 함수 - 전처리
def preprocess_function(example):
    # T5모델에 맞게 토큰화
    inputs = [f"Review: {text}\nSentiment (positive or negative)" for text in example['text']]
    model_input = tokenizer(inputs, max_length=128, truncation=True)
    # 라벨 토큰화
    labels = tokenizer(text_target=example['label'], max_length=128, truncation=True)
    model_input['labels'] = labels['input_ids']
    return model_input
# 2. DataSet 객체
from datasets import Dataset
train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

# 3. 토큰화 매핑
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_test = test_dataset.map(preprocess_function, batched=True, remove_columns=test_dataset.column_names)

Map: 100%|██████████| 5/5 [00:00<00:00, 1249.64 examples/s]


In [10]:
# Training Arguments 설정
training_args = Seq2SeqTrainingArguments(
    output_dir='./t5_sentiment_results',
    eval_strategy='epoch',
    learning_rate=3e-4,
    num_train_epochs=5,
    predict_with_generate=True,
    logging_steps=2
) 
# Data Collator, Trainer 객체
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
trainer = Seq2SeqTrainer(
    model=model, args=training_args, train_dataset=tokenized_train, eval_dataset=tokenized_test,
    processing_class=tokenizer, data_collator = data_collator
)
# 학습시간 측정
start_time = time.time()
trainer.train()
training_time = time.time() - start_time
print(f"Fine-tuning completed time : {training_time:.2f} seconds")

c:\Users\Playdata\miniconda3\envs\llm\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,1.356085,0.010005
2,0.007457,0.009080
3,0.069067,0.009102
4,0.001030,0.009404
5,0.002056,0.009494


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.72it/s]
c:\Users\Playdata\miniconda3\envs\llm\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Fine-tuning completed time : 6.09 seconds


#### 파인튜닝된 모델 최종 평가

In [13]:
ft_acc, ft_df = evaluate_model(model, tokenizer, test_data, "fine_tuned")
print(f"Fine-tuned Model Accuracy: {ft_acc:.2%}")
print(ft_df[["Text", "True Label", "Pred Label", "Correct"]].to_string())

Fine-tuned Model Accuracy: 100.00%
                                                                 Text True Label Pred Label  Correct
0             The movie was mediocre, but the ending was spectacular!   positive   positive     True
1              Worst movie I have seen this year. Avoid at all costs.   negative   negative     True
2  A refreshing and delightful comedy that had me laughing all night.   positive   positive     True
3           The acting was flat and the script felt extremely forced.   negative   negative     True
4                 An absolute gem of a film with outstanding visuals.   positive   positive     True
